# 02 Feature EDA

`d`, `C`, `WBGT`, `J_i` を中心に、分布・相関・カテゴリ比率を見ます。

実データがまだ無くても、対象フレームの選択と探索までは動くようにしています。

In [10]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

try:
    import matplotlib.pyplot as plt
except Exception:  # pragma: no cover - notebook fallback
    plt = None

try:
    from IPython.display import display
except Exception:  # pragma: no cover - notebook fallback
    display = print

REPO_ROOT = Path.cwd().resolve()
for candidate in [REPO_ROOT, *REPO_ROOT.parents]:
    if (candidate / 'data').exists() and (candidate / 'src').exists():
        REPO_ROOT = candidate
        break

DATA_DIRS = [REPO_ROOT / 'data' / 'samples', REPO_ROOT / 'data' / 'processed']
SUPPORTED_SUFFIXES = {'.parquet', '.csv', '.json', '.geojson'}

def read_geojson(path: Path) -> pd.DataFrame:
    payload = json.loads(path.read_text(encoding='utf-8'))
    features = payload.get('features', []) if isinstance(payload, dict) else []
    rows: list[dict[str, object]] = []
    for feature in features:
        properties = dict(feature.get('properties') or {})
        geometry = feature.get('geometry') or {}
        properties['geometry_type'] = geometry.get('type')
        properties['geometry'] = geometry.get('coordinates')
        rows.append(properties)
    return pd.DataFrame(rows)

def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix == '.parquet':
        return pd.read_parquet(path)
    if suffix == '.csv':
        return pd.read_csv(path)
    if suffix in {'.json', '.geojson'}:
        return read_geojson(path)
    raise ValueError(f'Unsupported file: {path}')

def discover_frames() -> dict[str, pd.DataFrame]:
    frames: dict[str, pd.DataFrame] = {}
    for directory in DATA_DIRS:
        if not directory.exists():
            continue
        for path in sorted(directory.rglob('*')):
            if path.is_file() and path.suffix.lower() in SUPPORTED_SUFFIXES:
                try:
                    frames[path.stem] = read_table(path)
                except Exception:
                    continue
    return frames

frames = discover_frames()
print(f'loaded {len(frames)} frame(s)')
for name, frame in frames.items():
    print(f'- {name}: {frame.shape}')

if plt is not None:
    plt.style.use('seaborn-v0_8-whitegrid')

loaded 0 frame(s)


In [11]:
def select_eda_frame(frames: dict[str, pd.DataFrame]) -> tuple[str | None, pd.DataFrame | None]:
    if not frames:
        return None, None
    priority_terms = ('feature', 'score', 'grid', 'sample', 'processed')
    ranked: list[tuple[tuple[int, int, int], str, pd.DataFrame]] = []
    for name, frame in frames.items():
        numeric_count = len(frame.select_dtypes(include='number').columns)
        preferred = int(any(term in name.lower() for term in priority_terms))
        ranked.append(((preferred, numeric_count, len(frame)), name, frame))
    ranked.sort(reverse=True)
    _, name, frame = ranked[0]
    return name, frame

frame_name, frame = select_eda_frame(frames)
if frame is None:
    print('No readable data found yet.')
else:
    print(f'EDA target: {frame_name}')
    display(frame.head(10))
    numeric_columns = frame.select_dtypes(include='number').columns.tolist()
    print('numeric columns:', numeric_columns)

    if plt is not None and numeric_columns:
        plot_columns = numeric_columns[:8]
        axes = frame[plot_columns].hist(figsize=(14, 8), bins=20, grid=False)
        if hasattr(axes, 'ravel'):
            axes_list = axes.ravel().tolist()
        else:
            axes_list = [axes]
        for axis in axes_list:
            axis.set_ylabel('count')
        plt.suptitle(f'Feature distributions: {frame_name}')
        plt.tight_layout()
        plt.show()

    if numeric_columns:
        desc = frame[numeric_columns].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]).T
        display(desc)

No readable data found yet.


In [12]:
if frame is not None:
    numeric_columns = frame.select_dtypes(include='number').columns.tolist()
    if plt is not None and len(numeric_columns) >= 2:
        corr = frame[numeric_columns].corr(numeric_only=True)
        fig, ax = plt.subplots(figsize=(max(8, len(numeric_columns) * 0.8), max(6, len(numeric_columns) * 0.6)))
        image = ax.imshow(corr, vmin=-1, vmax=1, cmap='RdBu_r')
        ax.set_xticks(range(len(numeric_columns)))
        ax.set_xticklabels(numeric_columns, rotation=45, ha='right')
        ax.set_yticks(range(len(numeric_columns)))
        ax.set_yticklabels(numeric_columns)
        fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04, label='correlation')
        ax.set_title(f'Correlation heatmap: {frame_name}')
        plt.tight_layout()
        plt.show()

    candidate_text_columns = [
        column for column in frame.columns
        if frame[column].dtype == 'object' or str(frame[column].dtype).startswith('category')
    ]
    for column in candidate_text_columns[:3]:
        counts = frame[column].value_counts(dropna=False).head(10)
        print(f'\nTop values for {column}')
        display(counts.to_frame('count'))

    time_like_columns = [column for column in frame.columns if any(token in column.lower() for token in ['time', 'date', 'hour', 'timestamp'])]
    for column in time_like_columns[:2]:
        parsed = pd.to_datetime(frame[column], errors='coerce')
        if parsed.notna().any():
            print(f'\nTime profile for {column}')
            display(parsed.dt.hour.value_counts(dropna=False).sort_index().to_frame('count'))

In [13]:
if frame is not None:
    print('EDA complete.')
    print('If this is not the intended target frame, adjust select_eda_frame().')